# Assignment #4: Soft Computer, Reworked with Hand-Tagged Semantics

reworking my first assignment (the soft computer poetry generator) to use our class's hand-tagged semantic corpus. instead of picking from small hardcoded vocabulary lists i wrote myself, each line now pulls from the nearest neighbors of a target vector in the shared tag-space. the structural scaffolding of the original poem stays, but the vocabulary comes from the class's collective sense of which words feel organic, warm, sharp, emotive, etc.

In [86]:
import json
import random
from collections import defaultdict, Counter
from simpleneighbors import SimpleNeighbors

In [87]:
%pip install simpleneighbors scikit-learn numpy

Note: you may need to restart the kernel to use updated packages.


In [88]:
# loading the tagged data
lines = open("semantic-categories-2026-04-10.jsonl").readlines()

cats = defaultdict(list)
for item in lines:
    obj = json.loads(item)
    if len(obj['label']) > 0:
        cats[obj['text'].lower()].extend(obj['label'])

In [89]:
# count tags per word
cats_counted = {word: Counter(tags) for word, tags in cats.items()}

In [90]:
# build the column order for vectors
unique_features = set()
for val in cats.values():
    unique_features.update(val)

cols = list(unique_features)
name2idx = {name: i for i, name in enumerate(cols)}
print(cols)

['cold', 'sharp', 'warm', 'tangible', 'organic', 'round', 'emotive', 'academic', 'outlandish', 'youthful']


In [91]:
# vectorize every word
cats_vectorized = {}
for word, counts in cats_counted.items():
    cats_vectorized[word] = [counts[feat] for feat in cols]

In [92]:
# build the nearest-neighbor lookup
lookup = SimpleNeighbors(len(cols), metric="angular")
for word, vec in cats_vectorized.items():
    lookup.add_one(word, vec)
lookup.build()

In [93]:
# helper to build target vectors by tag name
def make_vector(**kwargs):
    vec = [0] * len(cols)
    for k, v in kwargs.items():
        vec[name2idx[k]] = v
    return vec

## nearest neighbors only
The main poem generator, each slot has a target vector describing the "vibe" I want - material aims for organic + tangible, power aims for warm + organic, inhabitant aims for emotive + warm. The generator picks a random word from the 20 nearest neighbors of each target.

In [94]:
# generate poem
stanza_count = 7

for i in range(stanza_count):
    material   = random.choice(lookup.nearest(make_vector(organic=2, tangible=2), n=20))
    structure  = random.choice(lookup.nearest(make_vector(round=2, warm=1), n=20))
    power      = random.choice(lookup.nearest(make_vector(warm=3, organic=1), n=20))
    interface  = random.choice(lookup.nearest(make_vector(tangible=2, sharp=1), n=20))
    location   = random.choice(lookup.nearest(make_vector(warm=2, round=2), n=20))
    inhabitant = random.choice(lookup.nearest(make_vector(emotive=3, warm=1), n=20))
    promise    = random.choice(lookup.nearest(make_vector(emotive=2, youthful=1), n=20))

    print()
    print(f"a computer of {material}")
    print(f"     bound by {structure}")
    print(f"          powered by {power}")
    print(f"                with {interface}")
    print(f"                     resting among {location}")
    print(f"                          inhabited by {inhabitant}")
    print(f"                               and it offers {promise}.")


a computer of threads
     bound by harmonious
          powered by afternoon
                with plow
                     resting among lamp
                          inhabited by degrading
                               and it offers near.

a computer of flute-player
     bound by breast
          powered by temperate
                with doorways
                     resting among shaft
                          inhabited by concerns
                               and it offers functionless.

a computer of posts
     bound by character
          powered by light
                with plow
                     resting among ease
                          inhabited by passion
                               and it offers functionless.

a computer of woman
     bound by loop
          powered by asleep
                with thing
                     resting among nod
                          inhabited by frail words
                               and it offers charming.

a computer o

## rejection
The first version still let in words tagged cold or sharp (hunger, gunpowder, straight). This version adds a rule: refuse any word tagged cold, sharp, or academic. The soft computer isn't those things, so the vocabulary shouldn't be either.

In [95]:
# helper to check if a word has forbidden tags
def has_forbidden(word, forbidden):
    """return True if the word was tagged with any forbidden category"""
    tags = set(cats_counted.get(word, Counter()).keys())
    return bool(tags & forbidden)

def pick_soft(target_vec, forbidden, pool_size=40, max_tries=20):
    """pick a word near target_vec that has none of the forbidden tags"""
    candidates = lookup.nearest(target_vec, n=pool_size)
    for _ in range(max_tries):
        word = random.choice(candidates)
        if not has_forbidden(word, forbidden):
            return word
    # if can't find a clean one, return the last candidate anyway
    return word

In [96]:
# generate, refusing cold/sharp/academic words
forbidden = {"cold", "sharp", "academic"}

stanza_count = 7

for i in range(stanza_count):
    material   = pick_soft(make_vector(organic=2, tangible=2), forbidden)
    structure  = pick_soft(make_vector(round=2, warm=1), forbidden)
    power      = pick_soft(make_vector(warm=3, organic=1), forbidden)
    interface  = pick_soft(make_vector(tangible=2), forbidden)
    location   = pick_soft(make_vector(warm=2, round=2), forbidden)
    inhabitant = pick_soft(make_vector(emotive=3, warm=1), forbidden)
    promise    = pick_soft(make_vector(emotive=2, youthful=1), forbidden)

    print()
    print(f"a computer of {material}")
    print(f"     bound by {structure}")
    print(f"          powered by {power}")
    print(f"                with {interface}")
    print(f"                     resting among {location}")
    print(f"                          inhabited by {inhabitant}")
    print(f"                               and it offers {promise}.")


a computer of gasoline
     bound by pockets
          powered by touch
                with asia
                     resting among whispers
                          inhabited by temper
                               and it offers forgivable.

a computer of woman
     bound by lovely
          powered by heaven
                with the pilot
                     resting among house
                          inhabited by concerns
                               and it offers beauty.

a computer of hoes
     bound by stamp
          powered by clothes
                with shelf
                     resting among okay
                          inhabited by wives
                               and it offers imagination.

a computer of gasoline
     bound by breast
          powered by share
                with bottle
                     resting among hearts
                          inhabited by wives
                               and it offers enchanted.

a computer of glass
     bou

## tag-descriptor
Each word is printed with its top two tags in parentheses, making the hand-tagging visible in the output itself. More of a formal study than a poem.

In [97]:
# helper to read top tags for a word
def top_tags(word, n=2):
    """return the n most-tagged categories for a word, as a list"""
    counts = cats_counted.get(word, Counter())
    return [tag for tag, _ in counts.most_common(n)]

In [98]:
# generate with tag descriptors
stanza_count = 7

for i in range(stanza_count):
    material   = random.choice(lookup.nearest(make_vector(organic=2, tangible=2), n=20))
    structure  = random.choice(lookup.nearest(make_vector(round=2, warm=1), n=20))
    power      = random.choice(lookup.nearest(make_vector(warm=3, organic=1), n=20))
    interface  = random.choice(lookup.nearest(make_vector(tangible=2, sharp=1), n=20))
    location   = random.choice(lookup.nearest(make_vector(warm=2, round=2), n=20))
    inhabitant = random.choice(lookup.nearest(make_vector(emotive=3, warm=1), n=20))
    promise    = random.choice(lookup.nearest(make_vector(emotive=2, youthful=1), n=20))

    def describe(word):
        tags = top_tags(word, 2)
        return f"{word} ({', '.join(tags)})" if tags else word

    print()
    print(f"a computer of {describe(material)}")
    print(f"     bound by {describe(structure)}")
    print(f"          powered by {describe(power)}")
    print(f"                with {describe(interface)}")
    print(f"                     resting among {describe(location)}")
    print(f"                          inhabited by {describe(inhabitant)}")
    print(f"                               and it offers {describe(promise)}.")


a computer of wood (organic, tangible)
     bound by full (round, emotive)
          powered by notions (warm)
                with workmen (tangible, sharp)
                     resting among coats (warm, round)
                          inhabited by temper (emotive)
                               and it offers _statirical_ (emotive, youthful).

a computer of men singers (organic, tangible)
     bound by lamp (round, warm)
          powered by yellow (warm, organic)
                with vehicle (tangible, sharp)
                     resting among workmanship (round, warm)
                          inhabited by temper (emotive)
                               and it offers near (emotive, youthful).

a computer of chestnut (organic, tangible)
     bound by again (round)
          powered by asleep (warm, emotive)
                with workmen (tangible, sharp)
                     resting among okay (round, warm)
                          inhabited by husband (emotive, warm)
            